In [1]:
import joblib
model = joblib.load(r"C:\Users\vishnu\Downloads\FINAL_model (1).pkl")

print(model)

C:\Users\vishnu\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\vishnu\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\vishnu\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.7.2 when using version 1.6.1. This might lead to bre

CalibratedClassifierCV(cv='prefit',
                       estimator=Pipeline(steps=[('preprocess',
                                                  ColumnTransformer(force_int_remainder_cols='deprecated',
                                                                    transformers=[('num',
                                                                                   StandardScaler(),
                                                                                   ['issued_amount',
                                                                                    'initial_interest_rate',
                                                                                    'initial_loan_duration',
                                                                                    'customer_risk_rating_was_missing']),
                                                                                  ('cat',
                                                                       

C:\Users\vishnu\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\vishnu\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\vishnu\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.7.2 when using version 1.6.1. This might lead to

In [2]:
# ============================================================
# MACRO OVERLAY & STRESS SCENARIOS
# ============================================================
#
# IMPORTANT FRAMING:
# This is scenario-based sensitivity analysis.
# It is NOT econometric estimation.
#
# The dataset does not contain borrower-level unemployment
# or inflation variables. Therefore, macroeconomic scenarios
# are externally imposed on the final model probabilities.
#
# The final model itself is NOT changed or retrained.
# ============================================================

import pandas as pd
import numpy as np
import joblib
import os

# ============================================================
# 1. FILE PATHS
# ============================================================

MODEL_PATH = r"C:\Users\vishnu\Downloads\FINAL_model (1).pkl"
X_TEST_PATH = r"C:\Users\vishnu\Downloads\X_test.csv"
Y_TEST_PATH = r"C:\Users\vishnu\Downloads\y_test.csv"

OUTPUT_LOAN_LEVEL = r"C:\Users\vishnu\Downloads\macro_stress_results.csv"
OUTPUT_SUMMARY = r"C:\Users\vishnu\Downloads\macro_stress_summary.csv"

In [3]:
# ============================================================
# 2. CHECK REQUIRED FILES
# ============================================================

print("=" * 70)
print("MACROECONOMIC STRESS TESTING")
print("=" * 70)

required_files = [
    MODEL_PATH,
    X_TEST_PATH,
    Y_TEST_PATH
]

for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required file not found: {path}\n"
            f"Please check the file path before running."
        )

print("\nAll required files found.")

MACROECONOMIC STRESS TESTING

All required files found.


In [4]:
# ============================================================
# 3. LOAD FINAL MODEL AND LOCKED TEST DATA
# ============================================================

print("\nLoading final model...")
model = joblib.load(MODEL_PATH)

print("Loading locked test set...")
X_test = pd.read_csv(X_TEST_PATH)
y_test = pd.read_csv(Y_TEST_PATH).squeeze()

print(f"X_test rows: {len(X_test):,}")
print(f"y_test rows: {len(y_test):,}")

assert len(X_test) == len(y_test), \
    "ERROR: X_test and y_test row counts do not match."

print("Test-set consistency check: PASSED")


Loading final model...
Loading locked test set...
X_test rows: 81,395
y_test rows: 81,395
Test-set consistency check: PASSED


In [5]:
# ============================================================
# 4. GENERATE BASELINE MODEL PROBABILITIES
# ============================================================

print("\nGenerating baseline model probabilities...")

baseline_prob = model.predict_proba(X_test)[:, 1]

print(f"Mean baseline predicted PD: {baseline_prob.mean():.4%}")
print(f"Median baseline predicted PD: {np.median(baseline_prob):.4%}")
print(f"Minimum baseline PD: {baseline_prob.min():.4%}")
print(f"Maximum baseline PD: {baseline_prob.max():.4%}")


Generating baseline model probabilities...
Mean baseline predicted PD: 11.1454%
Median baseline predicted PD: 9.2939%
Minimum baseline PD: 0.0000%
Maximum baseline PD: 49.6073%


In [6]:
# ============================================================
# 5. DEFINE MACRO STRESS SCENARIOS
# ============================================================
#
# IMPORTANT:
# These are externally imposed scenario assumptions.
#
# They are NOT coefficients estimated from the loan data.
#
# BASELINE:
# No macroeconomic deterioration.
#
# MODERATE:
# +1 percentage point unemployment
# +2 percentage points inflation
#
# SEVERE:
# +3 percentage points unemployment
# +4 percentage points inflation
#
# The severe scenario is designed as a 2020-shock-inspired
# adverse scenario, NOT as a claim that these exact changes
# represent the observed European 2020 values.
# ============================================================

SCENARIOS = {
    "Baseline": {
        "unemployment_change_pp": 0.0,
        "inflation_change_pp": 0.0
    },

    "Moderate Stress": {
        "unemployment_change_pp": 1.0,
        "inflation_change_pp": 2.0
    },

    "Severe Stress": {
        "unemployment_change_pp": 3.0,
        "inflation_change_pp": 4.0
    }
}

In [7]:
# ============================================================
# 6. SCENARIO SENSITIVITY ASSUMPTIONS
# ============================================================
#
# These are sensitivity assumptions.
# They are NOT statistically estimated relationships.
#
# For every +1 percentage-point unemployment shock:
# default odds increase by 10%.
#
# For every +1 percentage-point inflation shock:
# default odds increase by 5%.
#
# Odds-based scaling is used rather than directly adding
# probability points.
# ============================================================

UNEMPLOYMENT_ODDS_SENSITIVITY = 0.10
INFLATION_ODDS_SENSITIVITY = 0.05

In [8]:
# ============================================================
# 7. FUNCTION TO APPLY MACRO STRESS
# ============================================================

def apply_macro_stress(
    probabilities,
    unemployment_change_pp,
    inflation_change_pp
):

    # Prevent division by zero / probability = 1
    eps = 1e-10

    p = np.clip(
        probabilities,
        eps,
        1 - eps
    )

    # Convert probability to odds
    odds = p / (1 - p)

    # Unemployment effect
    unemployment_multiplier = (
        1
        + UNEMPLOYMENT_ODDS_SENSITIVITY
        * unemployment_change_pp
    )

    # Inflation effect
    inflation_multiplier = (
        1
        + INFLATION_ODDS_SENSITIVITY
        * inflation_change_pp
    )

    # Combined macro multiplier
    total_multiplier = (
        unemployment_multiplier
        * inflation_multiplier
    )

    # Apply stress to odds
    stressed_odds = odds * total_multiplier

    # Convert odds back to probability
    stressed_probability = (
        stressed_odds /
        (1 + stressed_odds)
    )

    return stressed_probability, total_multiplier

In [9]:
# ============================================================
# 8. APPLY ALL SCENARIOS
# ============================================================

results = pd.DataFrame(index=X_test.index)

results["baseline_probability"] = baseline_prob

scenario_multipliers = {}

for scenario_name, assumptions in SCENARIOS.items():

    stressed_probability, multiplier = apply_macro_stress(
        baseline_prob,
        assumptions["unemployment_change_pp"],
        assumptions["inflation_change_pp"]
    )

    column_name = (
        scenario_name.lower()
        .replace(" ", "_")
        + "_probability"
    )

    results[column_name] = stressed_probability

    scenario_multipliers[scenario_name] = multiplier

In [10]:
# ============================================================
# 9. ADD ACTUAL TEST OUTCOME
# ============================================================

results["actual_default"] = y_test.values

In [11]:
# ============================================================
# 10. SANITY CHECK 1
# MODERATE MUST BE GREATER THAN BASELINE
# ============================================================

baseline = results["baseline_probability"]
moderate = results["moderate_stress_probability"]
severe = results["severe_stress_probability"]

moderate_not_higher = (
    moderate <= baseline
).sum()

print("\n" + "=" * 70)
print("SANITY CHECK 1")
print("=" * 70)

print(
    "Rows where Moderate Stress <= Baseline:",
    f"{moderate_not_higher:,}"
)

if moderate_not_higher > 0:
    raise ValueError(
        "SANITY CHECK FAILED: Moderate stress is not "
        "higher than baseline for all observations."
    )

print("PASS: Moderate Stress > Baseline")


SANITY CHECK 1
Rows where Moderate Stress <= Baseline: 0
PASS: Moderate Stress > Baseline


In [12]:
# ============================================================
# 11. SANITY CHECK 2
# SEVERE MUST BE GREATER THAN BASELINE
# ============================================================

severe_not_higher = (
    severe <= baseline
).sum()

print("\n" + "=" * 70)
print("SANITY CHECK 2")
print("=" * 70)

print(
    "Rows where Severe Stress <= Baseline:",
    f"{severe_not_higher:,}"
)

if severe_not_higher > 0:
    raise ValueError(
        "SANITY CHECK FAILED: Severe stress is not "
        "higher than baseline for all observations."
    )

print("PASS: Severe Stress > Baseline")


SANITY CHECK 2
Rows where Severe Stress <= Baseline: 0
PASS: Severe Stress > Baseline


In [13]:
# ============================================================
# 12. SANITY CHECK 3
# SEVERE MUST BE GREATER THAN MODERATE
# ============================================================

severe_not_above_moderate = (
    severe <= moderate
).sum()

print("\n" + "=" * 70)
print("SANITY CHECK 3")
print("=" * 70)

print(
    "Rows where Severe Stress <= Moderate Stress:",
    f"{severe_not_above_moderate:,}"
)

if severe_not_above_moderate > 0:
    raise ValueError(
        "SANITY CHECK FAILED: Severe stress is not "
        "higher than moderate stress."
    )

print("PASS: Severe Stress > Moderate Stress")


SANITY CHECK 3
Rows where Severe Stress <= Moderate Stress: 0
PASS: Severe Stress > Moderate Stress


In [14]:
# ============================================================
# 13. PORTFOLIO-LEVEL SUMMARY
# ============================================================

summary_rows = []

for scenario_name, assumptions in SCENARIOS.items():

    if scenario_name == "Baseline":
        probs = baseline

    elif scenario_name == "Moderate Stress":
        probs = moderate

    elif scenario_name == "Severe Stress":
        probs = severe

    summary_rows.append({

        "scenario":
            scenario_name,

        "unemployment_change_pp":
            assumptions["unemployment_change_pp"],

        "inflation_change_pp":
            assumptions["inflation_change_pp"],

        "odds_multiplier":
            scenario_multipliers[scenario_name],

        "mean_predicted_default_probability":
            probs.mean(),

        "median_predicted_default_probability":
            np.median(probs),

        "p90_predicted_default_probability":
            np.percentile(probs, 90),

        "p95_predicted_default_probability":
            np.percentile(probs, 95),

        "number_of_loans":
            len(probs)
    })

summary = pd.DataFrame(summary_rows)

In [15]:
# ============================================================
# 14. CHANGE RELATIVE TO BASELINE
# ============================================================

baseline_mean = summary.loc[
    summary["scenario"] == "Baseline",
    "mean_predicted_default_probability"
].iloc[0]

summary["increase_vs_baseline_pp"] = (
    summary["mean_predicted_default_probability"]
    - baseline_mean
) * 100

summary["relative_increase_vs_baseline_pct"] = (
    (
        summary["mean_predicted_default_probability"]
        / baseline_mean
    ) - 1
) * 100

In [16]:
# ============================================================
# 15. PRINT FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("MACRO STRESS-TEST RESULTS")
print("=" * 70)

display_columns = [
    "scenario",
    "unemployment_change_pp",
    "inflation_change_pp",
    "odds_multiplier",
    "mean_predicted_default_probability",
    "increase_vs_baseline_pp",
    "relative_increase_vs_baseline_pct"
]

print(
    summary[display_columns].to_string(
        index=False
    )
)


MACRO STRESS-TEST RESULTS
       scenario  unemployment_change_pp  inflation_change_pp  odds_multiplier  mean_predicted_default_probability  increase_vs_baseline_pp  relative_increase_vs_baseline_pct
       Baseline                     0.0                  0.0             1.00                            0.111454                 0.000000                           0.000000
Moderate Stress                     1.0                  2.0             1.21                            0.130717                 1.926310                          17.283423
  Severe Stress                     3.0                  4.0             1.56                            0.160540                 4.908537                          44.040851


In [17]:
# ============================================================
# 16. PRINT RISK INTERPRETATION NUMBERS
# ============================================================

moderate_mean = moderate.mean()
severe_mean = severe.mean()

moderate_increase_pp = (
    moderate_mean - baseline_mean
) * 100

severe_increase_pp = (
    severe_mean - baseline_mean
) * 100

moderate_relative = (
    moderate_mean / baseline_mean - 1
) * 100

severe_relative = (
    severe_mean / baseline_mean - 1
) * 100

print("\n" + "=" * 70)
print("INTERPRETATION NUMBERS")
print("=" * 70)

print(
    f"Baseline mean PD: "
    f"{baseline_mean:.4%}"
)

print(
    f"Moderate mean PD: "
    f"{moderate_mean:.4%}"
)

print(
    f"Severe mean PD: "
    f"{severe_mean:.4%}"
)

print(
    f"\nModerate increase vs baseline: "
    f"{moderate_increase_pp:.2f} percentage points"
)

print(
    f"Moderate relative increase: "
    f"{moderate_relative:.2f}%"
)

print(
    f"\nSevere increase vs baseline: "
    f"{severe_increase_pp:.2f} percentage points"
)

print(
    f"Severe relative increase: "
    f"{severe_relative:.2f}%"
)


INTERPRETATION NUMBERS
Baseline mean PD: 11.1454%
Moderate mean PD: 13.0717%
Severe mean PD: 16.0540%

Moderate increase vs baseline: 1.93 percentage points
Moderate relative increase: 17.28%

Severe increase vs baseline: 4.91 percentage points
Severe relative increase: 44.04%


In [18]:
# ============================================================
# 17. SAVE RESULTS
# ============================================================

import os

os.makedirs(os.path.dirname(OUTPUT_LOAN_LEVEL), exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_SUMMARY), exist_ok=True)
results.to_csv(
    OUTPUT_LOAN_LEVEL,
    index=False
)

summary.to_csv(
    OUTPUT_SUMMARY,
    index=False
)

print("\n" + "=" * 70)
print("OUTPUT FILES")
print("=" * 70)

print(
    "Loan-level file:",
    OUTPUT_LOAN_LEVEL
)

print(
    "Portfolio summary:",
    OUTPUT_SUMMARY
)


OUTPUT FILES
Loan-level file: C:\Users\vishnu\Downloads\macro_stress_results.csv
Portfolio summary: C:\Users\vishnu\Downloads\macro_stress_summary.csv


In [19]:
# ============================================================
# 18. FINAL CHECK
# ============================================================

assert (
    baseline_mean
    < moderate_mean
    < severe_mean
)

print("\nFINAL CHECK PASSED")
print("Baseline < Moderate Stress < Severe Stress")

print("\nPerson B macro stress-testing completed.")


FINAL CHECK PASSED
Baseline < Moderate Stress < Severe Stress

Person B macro stress-testing completed.
